🪖 Bike Helmet Detection using YOLOv8

This project focuses on detecting whether bike riders are wearing helmets using a deep learning–based object detection model.  
The goal is to assist in enforcing road safety regulations by automatically identifying *helmet* and *no-helmet* cases from images or video feeds.  
The project falls under the **Automotive Safety** domain and demonstrates the power of **AI/ML in real-world surveillance systems**.


The Importing Section

In [10]:
!pip -q install ultralytics==8.3.50 matplotlib==3.9.0
import os, yaml, glob
from pathlib import Path
import ultralytics, torch
print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


ultralytics: 8.3.50
torch: 2.8.0+cu126
CUDA available: True


The Dataset - Processing

In [2]:
from google.colab import files
import zipfile, io, shutil

up = files.upload()  # choose your dataset zip
zip_name = list(up.keys())[0]

# Extract to /content/datasets
root = Path("/content/datasets")
root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(up[zip_name])) as z:
    z.extractall(root)

# Try to find the dataset root (folder that contains train/ and valid/)
candidates = []
for p in root.rglob("*"):
    if p.is_dir() and (p / "train").exists() and (p / "valid").exists():
        candidates.append(p)

if not candidates:
    # maybe the zip itself extracted as .../your_dataset/train
    raise SystemExit("Could not find 'train' and 'valid' folders. Check your ZIP structure.")
DATA_ROOT = sorted(candidates, key=lambda p: len(str(p)))[0]
print("DATA_ROOT:", DATA_ROOT)
assert (DATA_ROOT/"train/images").exists() and (DATA_ROOT/"train/labels").exists(), "train images/labels missing"
assert (DATA_ROOT/"valid/images").exists() and (DATA_ROOT/"valid/labels").exists(), "valid images/labels missing"
print("Found train/ and valid/ subfolders ✅")

# test is optional
has_test = (DATA_ROOT/"test/images").exists() and (DATA_ROOT/"test/labels").exists()
print("Has test set:", has_test)


Saving dataset.zip to dataset.zip
DATA_ROOT: /content/datasets/data
Found train/ and valid/ subfolders ✅
Has test set: True


Creating the yaml file

In [3]:
# Default classes (change as needed)
NAMES = ["helmet", "no-helmet"]   # <-- edit if your classes differ

DATA_YAML = "/content/helmet_data.yaml"
data_cfg = {
    "path": str(DATA_ROOT),                # not strictly required, but helpful
    "train": "train/images",               # your 'train' stays train/
    "val":   "valid/images",               # map your 'valid' → 'val' key
    "test":  "test/images" if has_test else "",
    "names": NAMES,
    "nc": len(NAMES),
}
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print(Path(DATA_YAML).read_text())


path: /content/datasets/data
train: train/images
val: valid/images
test: test/images
names:
- helmet
- no-helmet
nc: 2



Training the Model Using YOLOV8

In [4]:
!yolo detect train model=yolov8n.pt data=/content/helmet_data.yaml \
  epochs=180 imgsz=896 batch=8 device=0 \
  project=/content/runs name=helmet_180epoch exist_ok=True \
  optimizer=AdamW lr0=0.0017 momentum=0.9 weight_decay=0.0005 \
  warmup_epochs=3.0 warmup_momentum=0.8 warmup_bias_lr=0.0 \
  box=7.5 cls=0.5 dfl=1.5 hsv_h=0.015 hsv_s=0.7 hsv_v=0.4 \
  translate=0.10 scale=0.50 fliplr=0.5 mosaic=1.0 close_mosaic=10 patience=80 \
  save_period=10


100% 6.25M/6.25M [00:00<00:00, 134MB/s]
New https://pypi.org/project/ultralytics/8.3.226 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.50 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/helmet_data.yaml, epochs=180, time=None, patience=80, batch=8, imgsz=896, save=True, save_period=10, cache=False, device=0, workers=8, project=/content/runs, name=helmet_180epoch, exist_ok=True, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, r

Extra Training to get More Accuracy

In [6]:
!yolo detect train model=/content/runs/helmet_180epoch/weights/best.pt \
  data=/content/helmet_data.yaml epochs=50 imgsz=1024 batch=8 device=0 \
  project=/content/runs name=helmet_ft_1024 exist_ok=True patience=30


New https://pypi.org/project/ultralytics/8.3.226 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.50 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=/content/runs/helmet_180epoch/weights/best.pt, data=/content/helmet_data.yaml, epochs=50, time=None, patience=30, batch=8, imgsz=1024, save=True, save_period=-1, cache=False, device=0, workers=8, project=/content/runs, name=helmet_ft_1024, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_m

Testing With Test Images

In [7]:
!yolo detect predict model=/content/runs/helmet_180epoch/weights/best.pt \
  source=/content/datasets/data/test/images imgsz=896 conf=0.25 \
  project=/content/runs name=helmet_test_preds exist_ok=True


Ultralytics 8.3.50 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 168 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

image 1/63 /content/datasets/data/test/images/BikesHelmets10_png.rf.6d0a6559208c543175619eced4446d99.jpg: 896x896 1 no-helmet, 11.2ms
image 2/63 /content/datasets/data/test/images/BikesHelmets134_png.rf.de69ecef3ae6e5e83f89f71bde79f114.jpg: 896x896 6 helmets, 1 no-helmet, 11.3ms
image 3/63 /content/datasets/data/test/images/BikesHelmets142_png.rf.9d299e9cff627adeca39b3d994040a85.jpg: 896x896 2 helmets, 1 no-helmet, 11.2ms
image 4/63 /content/datasets/data/test/images/BikesHelmets145_png.rf.c0cfa39072714ec0dcbb643b80d8c1e6.jpg: 896x896 3 helmets, 11.2ms
image 5/63 /content/datasets/data/test/images/BikesHelmets149_png.rf.5dd3b9a8b187cb03c51196c2add5eba3.jpg: 896x896 1 helmet, 1 no-helmet, 11.2ms
image 6/63 /content/datasets/data/test/images/BikesHelmets14_png.rf.696772056ca1da3c52a31c0acf6c0140.jpg: 896x896 2 helmets, 11.2ms
